# Stage B1 — Architecture Search

Dual-fuel PINN pipeline | Sandrine Schueller Mafra | PPGEM – UFPR
Supports dissertation Section 3.2.1.2 (Table 6), feedback item asking
to connect Table 6 to a design-of-experiments framing and to justify
the winning architecture in terms of Occam's razor.

Self-contained: reloads and re-splits `masters_data.xlsx` exactly as A3
does (same code, repeated here) rather than depending on A3 having
been run first.

**Framework note:** this stage trains actual neural networks, so it
needs `tensorflow`/`keras` — `polars`/`plotly` alone can't do that.
Everything else (data handling, plotting) stays in polars/plotly as in
A1-A3.

**No physics here yet.** The candidates below are plain MLPs — `tanh`
hidden layers, MSE loss, nothing else — with the output head fixed for
every candidate: 4 linear emission outputs plus the η output built as in
A3 Section 9 (sigmoid on the physical fraction + fixed `Rescaling`,
Sec. 3.2.1.4). That is deliberate: Table 6 / Sec. 3.2.1 is the
*baseline* architecture search. The physics constraints (Table 7,
Eq. 3.9–3.15) and the composite physics-informed loss (Eq. 3.16) come
in **Stages C1 (constraints) and C2 (loss assembly)**, not this one.
What B1 selects is the backbone *shape* (layers/neurons) that both B2
(baseline, trained exactly as here) and C3/C4 (same shape, physics loss
added on top) will reuse — the architecture search and the physics are
deliberately separated so "does the shape help" and "does the physics
help" can be answered independently instead of tangled together.

**Input:** `data/masters_data.xlsx`
**What this notebook does:**
1. Rebuilds the OFAT-blocked, extremity-stratified split and train-only
   normalization from A3.
2. Defines the 12 candidate architectures from **Table 6**, plus **4
   proposed additions** (Section 2b — not in the dissertation text,
   reasoning given there), and checks every *reported* Table 6
   parameter count against the formula and against what Keras actually
   builds.
3. Runs 5-fold cross-validation × 3 seeds (15 fits per architecture,
   **240 fits total** across all 16) over the 34 train+val points —
   **test stays held out**, untouched until Phase D. Every fit is logged
   (overall and per-output MSE, R², train MSE, epochs run, best epoch).
4. Picks a winner not by lowest error alone, but by the *simplest*
   architecture that isn't statistically distinguishable from the best
   one (paired Wilcoxon test) — the Occam's-razor criterion the text
   already commits to in words — and records the whole decision in one
   decision log.

**Output:** every figure and result table is saved to `outputs/html/` as
`B1_<section>[_qualifier].html` (the decision log is
`B1_08_decision_log.html`). The raw per-fit results, the summary and the
decision log are also saved as CSV, and `B1_selected_architecture.json`
is written for B2 and C3/C4 (those files are read by code, so they stay
CSV/JSON).

**Runtime:** 240 small model fits on 34 points each — expect several
minutes, not seconds. Reduce `SEEDS` to `[0]` for a quick smoke test
before running the full search.

## Setup

In [ ]:
import json
import numpy as np
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from sklearn.model_selection import KFold
from scipy.stats import wilcoxon
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tqdm import tqdm

print("polars    ", pl.__version__)
import plotly
print("plotly    ", plotly.__version__)
print("tensorflow", tf.__version__)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "masters_data.xlsx"
OUT_DIR = PROJECT_ROOT / "outputs"
SEED = 42
RAW_PATH

## Color palette and output naming (shared across the whole pipeline)

`SPLIT_COLORS` (semantic role: train / validation / test / reference
value / alert / neutral) and `VARIABLE_COLORS` (identity of each of the
4 inputs and 5 outputs) are identical in every A/B/C notebook, so the
same element always has the same color in any chart of the pipeline.

Every figure or result table generated below is also saved to
`outputs/html/`, named `B1_<section>[_qualifier].html` — the number
matches the corresponding section header, so the order in which each
output was produced can be read from the file name alone.

In [ ]:
SPLIT_COLORS = {
    "train": "#B7C9DA",
    "validation": "#2B6EFF",
    "test": "#571D99",
    "reference": "#343A40",   # value transcribed from the dissertation text
    "alert": "#E85D04",       # outlier / out of range / anomaly
    "neutral": "#B0AFA8",     # grid lines / neutral reference
}
VARIABLE_COLORS = {
    "SOI": "#073b3a", "lambda": "#0b6e4f", "sub_rate": "#08a045", "P_rail": "#6bbf59",
    "NOx": "#c7adff", "PM": "#916dd5", "eta": "#7151a9", "HC": "#573d7f", "CO2": "#46325d",
}

HTML_DIR = OUT_DIR / "html"
HTML_DIR.mkdir(parents=True, exist_ok=True)


def flagged_table_html(df, title, out_path, flag_col=None, is_flagged=lambda v: False, ref_cols=()):
    """Result table -> Plotly go.Table -> HTML.
    Columns listed in ref_cols get the 'reference' tone in the header
    (values transcribed from the dissertation text). Cells in flag_col
    get the 'alert' tone wherever is_flagged(value) is True."""
    cols = list(df.columns)
    n = df.shape[0]
    header_fill = [SPLIT_COLORS["reference"] if c in ref_cols else "#F1F3F5" for c in cols]
    header_font = ["white" if c in ref_cols else "black" for c in cols]
    cell_fill = []
    for c in cols:
        if c == flag_col:
            cell_fill.append([SPLIT_COLORS["alert"] if is_flagged(v) else "white"
                               for v in df[c].to_list()])
        else:
            cell_fill.append(["white"] * n)
    fig = go.Figure(data=[go.Table(
        header=dict(values=cols, fill_color=header_fill,
                     font=dict(color=header_font), align="left"),
        cells=dict(values=[df[c].to_list() for c in cols],
                    fill_color=cell_fill, align="left"),
    )])
    fig.update_layout(title=title, margin=dict(t=40, l=10, r=10, b=10))
    fig.write_html(str(out_path), include_plotlyjs="inline")
    return fig


def simple_table_html(df, title, out_path):
    return flagged_table_html(df, title, out_path)

SPLIT_COLOR_OF = {"train": SPLIT_COLORS["train"], "val": SPLIT_COLORS["validation"],
                  "test": SPLIT_COLORS["test"]}

## 1. Load, split, normalize (same logic as A3)

Repeated here rather than imported so this notebook runs standalone.
See A3 for the extremity-based split rationale.

In [ ]:
COLUMN_MAP = {
    "SOI [o.CA]": "SOI", "Lambda [-]": "lambda", "Sub. Rate [%]": "sub_rate",
    "Prail [bar]": "P_rail", "HC [g/kW.h]": "HC", "NOX [ppm]": "NOx",
    "CO2 [%]": "CO2", "SO_H [FSN]": "PM", "ETA [%]": "eta",
}
INPUT_COLS = ["SOI", "lambda", "sub_rate", "P_rail"]
OUTPUT_COLS = ["HC", "NOx", "CO2", "PM", "eta"]
ALL_COLS = INPUT_COLS + OUTPUT_COLS

df = pl.read_excel(RAW_PATH).rename(COLUMN_MAP).select(ALL_COLS)
n = df.shape[0]

# OFAT block (median-deviation + smoothing)
medians = {c: df[c].median() for c in INPUT_COLS}
ranges = {c: (df[c].max() - df[c].min()) for c in INPUT_COLS}
deviation = np.column_stack([np.abs(df[c].to_numpy() - medians[c]) / ranges[c] for c in INPUT_COLS])
raw_block = np.array(INPUT_COLS)[deviation.argmax(axis=1)]

def smooth_isolated_labels(labels, passes=2):
    out = list(labels)
    for _ in range(passes):
        changed = False
        for i in range(1, len(out) - 1):
            if out[i] != out[i - 1] and out[i - 1] == out[i + 1]:
                out[i] = out[i - 1]
                changed = True
        if not changed:
            break
    return np.array(out)

ofat_block = smooth_isolated_labels(raw_block)

# extremity-based split
extremity = np.zeros(n)
for b in np.unique(ofat_block):
    idx = np.where(ofat_block == b)[0]
    vals = df[b].to_numpy()[idx]
    order = np.argsort(vals)
    m = len(idx)
    pos = np.array([0.5]) if m == 1 else np.empty(m)
    if m > 1:
        ranks = np.empty(m)
        ranks[order] = np.arange(m)
        pos = ranks / (m - 1)
    extremity[idx] = np.abs(pos - 0.5) * 2

rng = np.random.default_rng(SEED)
jitter = rng.uniform(-1e-9, 1e-9, size=n)
order = np.argsort(-(extremity + jitter))
split = np.array(["train"] * n)
split[order[:6]] = "test"
split[order[6:12]] = "val"
df = df.with_columns(pl.Series("split", split))

# min-max normalization fit on train only
train = df.filter(pl.col("split") == "train")
train_min = {c: train[c].min() for c in ALL_COLS}
train_max = {c: train[c].max() for c in ALL_COLS}
df = df.with_columns([
    ((pl.col(c) - train_min[c]) / (train_max[c] - train_min[c])).alias(f"{c}_norm")
    for c in ALL_COLS
])

print(dict(zip(*np.unique(split, return_counts=True))))
df.head(3)


## 2a. Candidate architectures from Table 6

Architecture is defined by hidden-layer widths alone; parameter count
is *computed*, not copied from the table, and compared against what
Table 6 states. Mismatches are flagged rather than assumed away — this
follows the same principle as A1's Table 4/5 check.

In [ ]:
ARCHITECTURES_TABLE6 = {
    "A1": (16,), "A2": (32,), "A3": (64,),
    "A4": (16, 16), "A5": (32, 32), "A6": (64, 64),
    "A7": (16, 8), "A8": (32, 16),
    "A9": (16, 16, 16), "A10": (32, 32, 32), "A11": (64, 32, 16),
    "A12": (16, 16, 16, 16),
}
TABLE6_PARAMS = {  # as printed in the dissertation text, Sec. 3.2.1.2
    "A1": 165, "A2": 325, "A3": 645, "A4": 421, "A5": 1317, "A6": 4933,
    "A7": 277, "A8": 725, "A9": 661, "A10": 2469, "A11": 3317, "A12": 949,
}
N_IN, N_OUT = len(INPUT_COLS), len(OUTPUT_COLS)

def count_params(hidden, in_dim=N_IN, out_dim=N_OUT):
    total, prev = 0, in_dim
    for h in hidden:
        total += prev * h + h
        prev = h
    total += prev * out_dim + out_dim
    return total

rows = []
for aid, hidden in ARCHITECTURES_TABLE6.items():
    computed = count_params(hidden)
    stated = TABLE6_PARAMS[aid]
    rows.append({
        "architecture": aid, "hidden_layers": len(hidden), "shape": str(hidden),
        "computed_params": computed, "table6_params": stated,
        "status": "match" if computed == stated else f"MISMATCH ({computed - stated:+d})",
    })
arch_table = pl.DataFrame(rows)
flagged_table_html(arch_table, "B1 -- Table 6 parameter counts: computed vs. text",
                    HTML_DIR / "B1_02_table6_param_check.html",
                    flag_col="status", is_flagged=lambda v: v != "match", ref_cols=("table6_params",))
arch_table

**Equation implemented above** — parameters of a fully-connected
stack with hidden widths $n_1, \dots, n_L$, input width $n_0=4$ and
output width $n_{L+1}=5$ (bias term per layer):

$$
N_{\text{params}} = \sum_{l=1}^{L+1} \left( n_{l-1} \cdot n_l + n_l \right)
$$

Shown against what Table 6 states, per architecture:

In [ ]:
mismatch = [s != "match" for s in arch_table["status"].to_list()]
fig = go.Figure()
fig.add_trace(go.Bar(x=arch_table["architecture"], y=arch_table["computed_params"],
                      name="computed (alert = differs from Table 6)",
                      marker_color=[SPLIT_COLORS["alert"] if m else SPLIT_COLORS["neutral"] for m in mismatch]))
fig.add_trace(go.Bar(x=arch_table["architecture"], y=arch_table["table6_params"],
                      name="Table 6 (text)", marker_color=SPLIT_COLORS["reference"]))
fig.update_layout(barmode="group", title="Parameter count: computed vs. Table 6",
                   yaxis_title="parameters", width=800, height=420)
fig.show()
fig.write_html(str(HTML_DIR / "B1_02_table6_param_bar.html"), include_plotlyjs="inline")

## 2b. Proposed additions — not in Table 6

Table 6's 12 candidates span 165 to 4805 parameters (computed — Table 6
prints 4933 for A6), i.e. 6x to 178x the ~27 points in a single training
fold, across 1-4 hidden layers and widths 8-64. Four gaps are worth
closing before trusting that grid, each for a specific reason — not a
blind "add more":

| id | shape | why |
|---|---|---|
| **A13_linear** | no hidden layer | Floor check. Every Table 6 candidate is already 6x+ over-parameterized for a 27-point training fold; worth confirming a model with no hidden layer isn't *already* competitive before trusting any nonlinear one. (The η output keeps its sigmoid head, so "linear" means no hidden layer, not a purely linear map for η.) |
| **A14_tiny4** | 1×4 | Table 6's smallest is 16 units (A1). Nothing tests the gap between that and no hidden layer — is there a smooth transition or a cliff? |
| **A15_tiny8** | 1×8 | Same purpose, one step up from A14. |
| **A16_wide128** | 1×128 | Ceiling check for a *single* layer: Table 6 stops at 64 (A3). Completes the width sweep 4→8→16→32→64→128 instead of assuming 64 was already the top of the useful range. |

**Deliberately not proposing:** deeper (5+ layer) or larger multi-layer
combinations. Table 6 already reaches 178x over-parameterized at A6;
going bigger multiplies runtime for a question the existing grid's own
results (Section 6 below) already lean against — more capacity hasn't
clearly helped so far, and 34 points won't support much more before
these become memorization exercises rather than fits.

In [ ]:
ARCHITECTURES_NEW = {
    "A13_linear": (),
    "A14_tiny4": (4,),
    "A15_tiny8": (8,),
    "A16_wide128": (128,),
}
ARCHITECTURES = {**ARCHITECTURES_TABLE6, **ARCHITECTURES_NEW}

new_rows = [{"architecture": aid, "hidden_layers": len(hidden), "shape": str(hidden) if hidden else "(none)",
             "computed_params": count_params(hidden)}
            for aid, hidden in ARCHITECTURES_NEW.items()]
proposed_table = pl.DataFrame(new_rows)
simple_table_html(proposed_table, "B1 -- Proposed additions (not in Table 6)",
                   HTML_DIR / "B1_02_proposed_additions.html")
proposed_table

## 3. Model builder & CV protocol

All 5 outputs predicted jointly (multi-output regression) from the 4
normalized inputs; `tanh` activation in every hidden layer, matching the
text's justification in Sec. 3.2.1.3 (bounded, smooth derivatives —
relevant again once Phase C needs gradients through this same
architecture). Test rows are excluded from the CV pool entirely.

**Equations implemented for every hidden layer $l$ and for the output head:**

$$
h^{(l)} = \tanh\!\left( W^{(l)} h^{(l-1)} + b^{(l)} \right), \qquad
\hat{y}_{\text{HC, NOx, CO2, PM}} = W_e h^{(L)} + b_e, \qquad
\hat{y}_{\eta} = \frac{\sigma\!\left(w_\eta^\top h^{(L)} + b_\eta\right) - \eta_{\min}}{\eta_{\max} - \eta_{\min}}
$$

The four emissions use a linear regression head. The η head is the one
fixed in A3 Section 9: a sigmoid on η as a physical fraction (bounded to
(0, 1), Sec. 3.2.1.4), followed by a non-trainable `Rescaling` with the
train-only η minimum/maximum, so the loss sees η on the same normalized
scale as the other outputs. The two heads together have exactly the
parameters of a single `Dense(5)` layer, so the counts of Section 2a are
unchanged (checked against Keras below).

**Implementation choice (not specified in the text):** the bias of the η
`Dense` layer is initialized to `logit(mean train η)`, so every fit starts
predicting the average efficiency instead of σ(0) = 0.5, which would sit
far outside the measured range (≈0.34–0.39) and dominate the loss in the
first epochs. It does not change the parameter count or what the model
can learn.

**Early stopping — difference from the text:** Sec. 3.2.1.2 specifies a
patience of 200 epochs. This search uses `PATIENCE = 30` (and at most
`EPOCHS = 300`) because it runs 240 fits; a patience of 200 would make
most fits run to the epoch cap and multiply the runtime without changing
the comparison between architectures, which is what this stage is for.
B2 retrains the selected architecture once with the text's patience of
200. The per-fit log in Section 4 records how many epochs each fit
actually ran and whether early stopping fired, so this choice can be
checked rather than assumed.

In [ ]:
assert OUTPUT_COLS[-1] == "eta", "the eta head is appended last; OUTPUT_COLS must end with eta"

# eta as a physical fraction (A1 Section 7: stored as a fraction) and its train-only range (A3 Section 9)
to_fraction = (lambda v: v / 100) if df["eta"].max() > 1 else (lambda v: v)
ETA_MIN, ETA_MAX = to_fraction(train_min["eta"]), to_fraction(train_max["eta"])
ETA_MEAN_TRAIN = float(np.mean(to_fraction(df.filter(pl.col("split") == "train")["eta"].to_numpy())))
ETA_BIAS_INIT = float(np.log(ETA_MEAN_TRAIN / (1 - ETA_MEAN_TRAIN)))    # logit(mean train eta)


def build_model(hidden_units, seed, input_dim=N_IN, output_dim=N_OUT):
    tf.random.set_seed(seed)
    inputs = keras.Input(shape=(input_dim,))
    x = inputs
    for units in hidden_units:
        x = layers.Dense(units, activation="tanh")(x)
    emissions = layers.Dense(output_dim - 1, activation="linear", name="emissions")(x)   # HC, NOx, CO2, PM
    eta_frac = layers.Dense(1, activation="sigmoid", name="eta_fraction",
                            bias_initializer=keras.initializers.Constant(ETA_BIAS_INIT))(x)
    eta_norm = layers.Rescaling(scale=1.0 / (ETA_MAX - ETA_MIN),
                                offset=-ETA_MIN / (ETA_MAX - ETA_MIN), name="eta_rescaled")(eta_frac)
    outputs = layers.Concatenate(name="outputs")([emissions, eta_norm])
    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer="adam", loss="mse")
    return model


def predict_np(model, X):
    """Forward pass as a NumPy array, without model.predict().

    model.predict() builds a new tf.function for every freshly built model;
    in a loop of 240 fits that triggers TensorFlow's "tf.function retracing"
    warning and is slower than a direct call on arrays this small. A direct
    call with training=False gives the same predictions (no dropout or batch
    normalization in these models)."""
    return np.asarray(model(np.asarray(X, dtype="float32"), training=False))

cv_pool = df.filter(pl.col("split") != "test")
X = cv_pool.select([f"{c}_norm" for c in INPUT_COLS]).to_numpy()
Y = cv_pool.select([f"{c}_norm" for c in OUTPUT_COLS]).to_numpy()
print("CV pool:", X.shape[0], "points (test held out separately)")
print(f"eta head: sigmoid on the fraction, Rescaling with eta_min={ETA_MIN:.6f}, eta_max={ETA_MAX:.6f}; "
      f"bias init = logit({ETA_MEAN_TRAIN:.4f}) = {ETA_BIAS_INIT:.4f}")

SEEDS = [0, 1, 2]
N_FOLDS = 5
EPOCHS = 300
PATIENCE = 30      # text (Sec. 3.2.1.2) says 200 -- see the note above
BATCH_SIZE = 8
ALPHA = 0.05       # significance level of the selection rule, Section 8

**Parameter count as built by Keras.** Every candidate is built once and
its `count_params()` compared with the formula of Section 2a — this
confirms that splitting the output into the emissions head + η head and
adding the `Rescaling` layer leaves the counts unchanged:

In [ ]:
kcheck_rows = []
for aid, hidden in ARCHITECTURES.items():
    keras.backend.clear_session()
    m = build_model(hidden, seed=0)
    kcheck_rows.append({"architecture": aid, "formula_params": count_params(hidden),
                        "keras_params": int(m.count_params()),
                        "matches": int(m.count_params()) == count_params(hidden)})
keras_param_check = pl.DataFrame(kcheck_rows)
flagged_table_html(keras_param_check, "B1 -- Parameter count: formula vs. Keras (with the eta head)",
                    HTML_DIR / "B1_03_keras_param_check.html",
                    flag_col="matches", is_flagged=lambda v: not v)
assert all(keras_param_check["matches"].to_list()), "Keras parameter count differs from the formula"
keras_param_check

**What the 5 folds actually look like**, for seed 0 (every architecture reuses these same splits, which is what makes the paired test in Section 8 valid):

In [ ]:
fold_map = np.full((N_FOLDS, X.shape[0]), "train", dtype=object)
kf0 = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEEDS[0])
for fold_i, (tr_idx, va_idx) in enumerate(kf0.split(X)):
    fold_map[fold_i, va_idx] = "validation"

z = (fold_map == "validation").astype(int)
fig = go.Figure(go.Heatmap(
    z=z, x=[f"pt {i}" for i in range(X.shape[0])], y=[f"fold {i}" for i in range(N_FOLDS)],
    colorscale=[[0, SPLIT_COLORS["train"]], [1, SPLIT_COLORS["validation"]]], showscale=False,
))
fig.update_layout(title="CV folds, seed 0: train (light) vs. validation (dark) points",
                   width=850, height=280, xaxis_showticklabels=False)
fig.show()
fig.write_html(str(HTML_DIR / "B1_03_fold_map.html"), include_plotlyjs="inline")

## 4. Run the search — 16 architectures × 3 seeds × 5 folds = 240 fits

This is the slow cell. `keras.backend.clear_session()` runs every
iteration to stop TensorFlow's graph/memory bookkeeping from growing
across 240 freshly-built models.

**Logged for every fit** (one row per architecture × seed × fold, saved
as `B1_cv_results_raw.csv` and shown in full in `B1_04_cv_fits_all.html`):
validation MSE and R² over all outputs, validation MSE per output, train
MSE at the restored best weights, the train–validation gap, the number
of epochs actually run, the best epoch, and whether early stopping fired
before the epoch cap.

In [ ]:
results = []
n_fits = len(ARCHITECTURES) * len(SEEDS) * N_FOLDS
with tqdm(total=n_fits, desc="B1 CV fits") as bar:
    for arch_id, hidden in ARCHITECTURES.items():
        for seed in SEEDS:
            kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
            for fold_i, (tr_idx, va_idx) in enumerate(kf.split(X)):
                keras.backend.clear_session()
                model = build_model(hidden, seed=seed * 100 + fold_i)
                es = keras.callbacks.EarlyStopping(
                    monitor="val_loss", patience=PATIENCE, restore_best_weights=True
                )
                hist = model.fit(
                    X[tr_idx], Y[tr_idx],
                    validation_data=(X[va_idx], Y[va_idx]),
                    epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0, callbacks=[es],
                )
                pred = predict_np(model, X[va_idx])
                pred_tr = predict_np(model, X[tr_idx])
                sq_err = (pred - Y[va_idx]) ** 2
                mse = float(np.mean(sq_err))
                ss_res = float(np.sum(sq_err))
                ss_tot = float(np.sum((Y[va_idx] - Y[va_idx].mean(axis=0)) ** 2))
                train_mse = float(np.mean((pred_tr - Y[tr_idx]) ** 2))
                epochs_run = len(hist.history["loss"])
                row = dict(architecture=arch_id, seed=seed, fold=fold_i,
                           n_train=len(tr_idx), n_val=len(va_idx),
                           mse=mse, r2=1 - ss_res / ss_tot, train_mse=train_mse,
                           gap=mse - train_mse, epochs_run=epochs_run,
                           best_epoch=int(np.argmin(hist.history["val_loss"])) + 1,
                           early_stopped=epochs_run < EPOCHS)
                for k, out in enumerate(OUTPUT_COLS):
                    row[f"mse_{out}"] = float(np.mean(sq_err[:, k]))
                results.append(row)
                bar.update(1)

cv_results = pl.DataFrame(results)
OUT_DIR.mkdir(parents=True, exist_ok=True)
cv_results.write_csv(OUT_DIR / "B1_cv_results_raw.csv")  # checkpoint, this cell is slow

fits_view = cv_results.with_columns([pl.col(c).round(5) for c in
                                     ["mse", "r2", "train_mse", "gap"] + [f"mse_{o}" for o in OUTPUT_COLS]])
flagged_table_html(fits_view, "B1 -- Every CV fit (240 rows); alert = early stopping never fired",
                    HTML_DIR / "B1_04_cv_fits_all.html",
                    flag_col="early_stopped", is_flagged=lambda v: not v)
cv_results.head()

## 5. Aggregate results per architecture

**Equations implemented per fold** (both computed on the normalized
outputs, over the held-out fold's $N$ points and all 5 outputs):

$$
\text{MSE} = \frac{1}{5N}\sum_{i=1}^{N}\sum_{k=1}^{5}\left(y_{i,k}-\hat y_{i,k}\right)^2,
\qquad
R^2 = 1 - \frac{\sum_{i,k}\left(y_{i,k}-\hat y_{i,k}\right)^2}{\sum_{i,k}\left(y_{i,k}-\bar y_{k}\right)^2}
$$

then averaged (mean ± std) over the 15 seed×fold fits per architecture.
The summary also carries the per-output validation MSE, the mean
train–validation gap (an overfitting indicator), the mean best epoch and
the share of fits in which early stopping fired. The heatmap after the
table shows which outputs drive each architecture's error.

In [ ]:
TRAIN_FOLD_SIZE = float(np.mean(cv_results["n_train"].to_numpy()))

summary_rows = []
for aid in ARCHITECTURES:
    sub = cv_results.filter(pl.col("architecture") == aid)
    mse_vals = sub["mse"].to_numpy()
    r2_vals = sub["r2"].to_numpy()
    row = {
        "architecture": aid,
        "source": "proposed" if aid in ARCHITECTURES_NEW else "Table 6",
        "n_params": count_params(ARCHITECTURES[aid]),
        "params_per_train_point": round(count_params(ARCHITECTURES[aid]) / TRAIN_FOLD_SIZE, 1),
        "mean_mse": mse_vals.mean(), "std_mse": mse_vals.std(ddof=1),
        "mean_r2": r2_vals.mean(), "std_r2": r2_vals.std(ddof=1),
        "mean_train_mse": float(sub["train_mse"].to_numpy().mean()),
        "mean_gap": float(sub["gap"].to_numpy().mean()),
        "mean_best_epoch": float(sub["best_epoch"].to_numpy().mean()),
        "early_stop_rate": float(np.mean(sub["early_stopped"].to_numpy())),
    }
    for out in OUTPUT_COLS:
        row[f"mean_mse_{out}"] = float(sub[f"mse_{out}"].to_numpy().mean())
    summary_rows.append(row)
summary = pl.DataFrame(summary_rows).sort("mean_mse")

summary_view = summary.with_columns([pl.col(c).round(5) for c in summary.columns
                                     if c.startswith("mean_") or c.startswith("std_")])
simple_table_html(summary_view, "B1 -- Summary per architecture (sorted by mean validation MSE)",
                   HTML_DIR / "B1_05_summary.html")
summary

In [ ]:
per_out = np.array([[summary[f"mean_mse_{o}"][i] for o in OUTPUT_COLS] for i in range(summary.shape[0])])
fig = go.Figure(go.Heatmap(
    z=per_out, x=OUTPUT_COLS, y=summary["architecture"].to_list(),
    colorscale=[[0, "white"], [1, SPLIT_COLORS["validation"]]],
    text=np.round(per_out, 4), texttemplate="%{text}", colorbar=dict(title="mean val MSE"),
))
fig.update_layout(title="Mean validation MSE per output and architecture (rows sorted by overall MSE)",
                   yaxis=dict(autorange="reversed"), width=750, height=620)
fig.show()
fig.write_html(str(HTML_DIR / "B1_05_per_output_mse_heatmap.html"), include_plotlyjs="inline")

## 6. Performance by architecture

In [ ]:
fig = go.Figure()
order_ids = summary["architecture"].to_list()
for aid in order_ids:
    vals = cv_results.filter(pl.col("architecture") == aid)["mse"].to_numpy()
    fig.add_trace(go.Box(y=vals, name=aid, marker_color=SPLIT_COLORS["validation"], boxpoints="all",
                          jitter=0.4, pointpos=0, showlegend=False))
fig.update_layout(title="Validation MSE across 15 CV fits (3 seeds x 5 folds), by architecture",
                   yaxis_title="MSE (normalized outputs)", xaxis_title="architecture",
                   width=850, height=450)
fig.show()
fig.write_html(str(HTML_DIR / "B1_06_mse_boxplot.html"), include_plotlyjs="inline")

## 7. Parameters vs. performance — the Occam's-razor view

If a small architecture sits at roughly the same height as a much
larger one, added complexity bought nothing. Diamonds are the four
proposed additions from Section 2b, circles are Table 6's original 12.
The x-axis is logarithmic because the candidates span two orders of
magnitude in size.

In [ ]:
is_new = [a in ARCHITECTURES_NEW for a in summary["architecture"].to_list()]
fig = go.Figure(go.Scatter(
    x=summary["n_params"].to_list(), y=summary["mean_mse"].to_list(),
    error_y=dict(type="data", array=summary["std_mse"].to_list(), visible=True,
                 color=SPLIT_COLORS["neutral"]),
    mode="markers+text", text=summary["architecture"].to_list(), textposition="top center",
    marker=dict(size=11, color=SPLIT_COLORS["validation"],
                symbol=["diamond" if f else "circle" for f in is_new],
                line=dict(color=SPLIT_COLORS["reference"], width=0.5)),
    showlegend=False,
))
fig.update_layout(title="Mean CV MSE vs. parameter count (diamonds = proposed, error bars = 1 std)",
                   xaxis=dict(title="parameters (log scale)", type="log"),
                   yaxis_title="mean validation MSE", width=800, height=480)
fig.show()
fig.write_html(str(HTML_DIR / "B1_07_params_vs_mse.html"), include_plotlyjs="inline")

## 8. Statistical comparison, final selection & decision log

The naive choice is whichever architecture has the lowest mean MSE.
Instead: take every architecture with *fewer* parameters than the
naive winner, run a paired Wilcoxon signed-rank test against it
(paired on matching seed+fold, since every architecture sees the same
15 CV splits), and if the difference isn't significant (p > `ALPHA`),
that simpler architecture is a legitimate substitute. Among the naive
winner and all such substitutes, keep the one with the fewest
parameters.

**Limits of this rule, worth stating in the text:**
- *Not significant is not the same as equivalent.* With 15 pairs the
  test has limited power, so a simpler architecture can be accepted
  because the test cannot tell them apart, not because they were shown
  to perform the same. An equivalence test (e.g. TOST with a stated
  margin) would be the stricter version of this rule.
- *The 15 pairs are not fully independent* — the folds of one seed share
  data, and the three seeds reshuffle the same 34 points — so the
  p-values are approximate.
- An architecture whose p-value sits close to `ALPHA` can switch side
  between runs; the decision log shows every p-value so this is visible.

The **decision log** below consolidates, for all 16 candidates: rank,
size, every KPI from Section 5, the p-value against the naive winner and
the resulting decision.

In [ ]:
naive_winner = summary["architecture"][0]
winner_params = summary["n_params"][0]
winner_vals = (cv_results.filter(pl.col("architecture") == naive_winner)
               .sort(["seed", "fold"])["mse"].to_numpy())

candidates = [(naive_winner, winner_params)]
comparisons = []
p_values = {}
for aid in summary["architecture"].to_list():
    if aid == naive_winner:
        continue
    p_count = count_params(ARCHITECTURES[aid])
    if p_count >= winner_params:
        continue
    vals = cv_results.filter(pl.col("architecture") == aid).sort(["seed", "fold"])["mse"].to_numpy()
    stat, p = wilcoxon(winner_vals, vals)
    p_values[aid] = float(p)
    comparisons.append({"architecture": aid, "n_params": p_count, "p_value": float(p),
                        "significant": bool(p <= ALPHA)})
    if p > ALPHA:
        candidates.append((aid, p_count))

selected = min(candidates, key=lambda t: t[1])

decisions = []
for rank, aid in enumerate(summary["architecture"].to_list(), start=1):
    if aid == selected[0] and aid == naive_winner:
        decision = "SELECTED (naive winner, no simpler substitute)"
    elif aid == selected[0]:
        decision = "SELECTED (simplest valid substitute)"
    elif aid == naive_winner:
        decision = "naive winner (lowest mean MSE), replaced by a simpler substitute"
    elif aid not in p_values:
        decision = "not tested (more parameters than the naive winner)"
    elif p_values[aid] > ALPHA:
        decision = "valid substitute, not the simplest"
    else:
        decision = "rejected (significantly worse than the naive winner)"
    decisions.append({"rank_by_mean_mse": rank, "architecture": aid,
                      "p_value_vs_naive_winner": round(p_values[aid], 4) if aid in p_values else None,
                      "decision": decision})

decision_log = (summary.select(["architecture", "source", "n_params", "params_per_train_point",
                                "mean_mse", "std_mse", "mean_r2", "std_r2", "mean_gap",
                                "mean_best_epoch", "early_stop_rate"])
                .with_columns([pl.col(c).round(5) for c in ["mean_mse", "std_mse", "mean_r2", "std_r2", "mean_gap"]]))
dec_df = pl.DataFrame(decisions)
decision_log = pl.DataFrame({**{c: dec_df[c].to_list() for c in ["rank_by_mean_mse"]},
                             **{c: decision_log[c].to_list() for c in decision_log.columns},
                             **{c: dec_df[c].to_list() for c in ["p_value_vs_naive_winner", "decision"]}})
decision_log.write_csv(OUT_DIR / "B1_decision_log.csv")
simple_table_html(decision_log,
                   f"B1 -- Decision log: rule = simplest architecture with p > {ALPHA} vs. the naive winner "
                   f"({naive_winner}); selected = {selected[0]}",
                   HTML_DIR / "B1_08_decision_log.html")

print(f"Naive winner (lowest mean MSE): {naive_winner}  (params={winner_params})")
print(f"SELECTED ARCHITECTURE: {selected[0]}  ({selected[1]} parameters)")
decision_log

**Same comparison, visually** — bars above the dashed line are
statistically indistinguishable from the naive winner, i.e. valid simpler
substitutes (full color); bars below it are significantly worse (faded).
The selected architecture is outlined.

In [ ]:
comp_df = pl.DataFrame(comparisons).sort("p_value", descending=True)
valid = [not s for s in comp_df["significant"].to_list()]
fig = go.Figure(go.Bar(
    x=comp_df["architecture"].to_list(), y=comp_df["p_value"].to_list(),
    marker=dict(color=SPLIT_COLORS["validation"], opacity=[1.0 if v else 0.35 for v in valid],
                line=dict(color=[SPLIT_COLORS["reference"] if a == selected[0] else SPLIT_COLORS["validation"]
                                 for a in comp_df["architecture"].to_list()],
                          width=[3 if a == selected[0] else 0 for a in comp_df["architecture"].to_list()])),
    text=[f"{p:.3f}" for p in comp_df["p_value"].to_list()], textposition="outside",
))
fig.add_hline(y=ALPHA, line_dash="dash", line_color=SPLIT_COLORS["reference"], annotation_text=f"p = {ALPHA}")
fig.update_layout(
    title=f"Paired Wilcoxon p-value vs. {naive_winner} (full color = valid simpler substitute; "
          f"outlined = selected, {selected[0]})",
    yaxis_title="p-value", xaxis_title="architecture (fewer params than the naive winner)",
    width=800, height=440,
)
fig.show()
fig.write_html(str(HTML_DIR / "B1_08_wilcoxon_pvalues.html"), include_plotlyjs="inline")

## Persist final outputs

Saves the full comparison table **and** the selected architecture's
identity — B2 reads the latter to know which shape to train, so run this
cell before starting B2. The JSON also records the η head constants and
the selection rule, so later stages can check they are building the same
model.

In [ ]:
summary.write_csv(OUT_DIR / "B1_architecture_summary.csv")

selection_record = {
    "architecture_id": selected[0],
    "hidden_units": list(ARCHITECTURES[selected[0]]),
    "n_params": selected[1],
    "is_proposed_addition": selected[0] in ARCHITECTURES_NEW,
    "naive_winner": naive_winner,
    "selection_rule": f"simplest architecture with paired Wilcoxon p > {ALPHA} vs. the naive winner",
    "p_value_vs_naive_winner": p_values.get(selected[0]),
    "output_head": {"emissions": "Dense(4, linear)",
                    "eta": "Dense(1, sigmoid) on the fraction + fixed Rescaling (A3 Section 9)",
                    "eta_min_train": ETA_MIN, "eta_max_train": ETA_MAX,
                    "eta_bias_init": ETA_BIAS_INIT},
    "cv_protocol": {"seeds": SEEDS, "n_folds": N_FOLDS, "epochs": EPOCHS,
                    "patience": PATIENCE, "batch_size": BATCH_SIZE},
}
with open(OUT_DIR / "B1_selected_architecture.json", "w") as f:
    json.dump(selection_record, f, indent=2)

print(f"Saved to {OUT_DIR}")
print(f"Selected architecture: {selected[0]} -> hidden layers {ARCHITECTURES[selected[0]]}")

## Next

**B2** trains the selected architecture once (no physics terms) on the
28-point train split, validated on the 6-point val split, with the
text's early-stopping patience of 200, as the baseline that Phase D
compares the PINN against — it reads `B1_selected_architecture.json`
from the cell above. The same architecture shape, with the same η head,
also becomes the starting point for **C3**'s hyperparameter search
(Table 8), which adds the physics-loss weighting on top.